In [1]:
# Gemma
from transformers import AutoProcessor, AutoModelForCausalLM

TARGET_MODEL_ID = "google/gemma-4-E2B-it"
ASSISTANT_MODEL_ID = "google/gemma-4-E2B-it-assistant"

# Target Model
processor = AutoProcessor.from_pretrained(TARGET_MODEL_ID)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID,
    dtype="auto",
    device_map="auto",

)

# Assistant Model (the drafter)
assistant_model = AutoModelForCausalLM.from_pretrained(
    ASSISTANT_MODEL_ID,
    dtype="auto",
    device_map="auto",
)
# Prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Write a short joke about saving RAM."},
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = target_model.generate(
    **inputs,
    assistant_model=assistant_model,
    max_new_tokens=256,
)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response)


/home/hamad/Desktop/Budget-office/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: Could not import module 'Gemma4Processor'. Are this object's requirements defined correctly?

In [25]:
from gliner import GLiNER

model = GLiNER.from_pretrained("gliner-community/gliner_small-v2.5")

text = """
شراء عبر نقاط البيع

بطاقة: ***7267;mada(Apple Pay)

لدى: BENZINA PETROL STAT

دولة: Bahrain

من: ***001

مبلغ: 13.700 BHD

في: 2026-08-29 11:51:44
"""

labels = ["amount", "date","store", "payment_method", "card_network","currency symbol"]
entities = model.predict_entities(text, labels, threshold=0.3)

for entity in entities:
    print(entity["text"], "=>", entity["label"])
print("Entities:", entities)

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 3055.03it/s]


7267 => currency symbol
mada => card_network
Apple Pay => payment_method
BENZINA PETROL => store
001 => currency symbol
مبلغ => currency symbol
13.700 => currency symbol
BHD => currency symbol
2026-08-29 => date
Entities: [{'start': 32, 'end': 36, 'text': '7267', 'label': 'currency symbol', 'score': 0.786089301109314}, {'start': 37, 'end': 41, 'text': 'mada', 'label': 'card_network', 'score': 0.7561705112457275}, {'start': 42, 'end': 51, 'text': 'Apple Pay', 'label': 'payment_method', 'score': 0.959656834602356}, {'start': 59, 'end': 73, 'text': 'BENZINA PETROL', 'label': 'store', 'score': 0.5339950919151306}, {'start': 102, 'end': 105, 'text': '001', 'label': 'currency symbol', 'score': 0.6742979288101196}, {'start': 107, 'end': 111, 'text': 'مبلغ', 'label': 'currency symbol', 'score': 0.326739102602005}, {'start': 113, 'end': 119, 'text': '13.700', 'label': 'currency symbol', 'score': 0.7339521050453186}, {'start': 120, 'end': 123, 'text': 'BHD', 'label': 'currency symbol', 'score': 

In [28]:
from app.routes.sms import extract_amount
amount = extract_amount(text)

ValueError: could not convert string to float: '13.700 BHD'